In [16]:
import numpy as np
from typing import Dict, Any, Sequence, Tuple

In [17]:
# ---------- Helper functions implementing formulas from the paper ----------
def compute_td_u(
    B_pu: float, 
    gamma_pu: float
) -> float:
    """
    TD,U = 1 / (Bp,u * log2(1 + gamma_pu))
    B_pu: bandwidth (bytes/sec) for DU p to user u (or effective bandwidth)
    gamma_pu: average SNR
    returns average latency (seconds) per byte
    """
    # prevent division by zero or log2(1+gamma)=0
    denom = B_pu * np.log2(1.0 + max(gamma_pu, 1e-12))
    if denom <= 0:
        return np.inf
    return 1.0 / denom

def compute_tm_u(
    td_u: float, 
    R_M_D: float, 
    U: int
) -> float:
    """
    TM,U = TD,U + 1 / (R_M,D / U) = TD,U + U / R_M,D
    R_M_D: total data rate from MEC to DU (bytes/sec)
    U: number of users sharing link equally
    returns average latency (seconds) per byte
    """
    if R_M_D <= 0 or U <= 0:
        return np.inf
    return td_u + (U / R_M_D)

def compute_tc_u(
    tm_u: float, 
    R_C_M: float, 
    U: int
) -> float:
    """
    TC,U = TM,U + 1 / (R_C,M / U) = TM,U + U / R_C,M
    R_C_M: total data rate from Cloud to MEC (bytes/sec)
    """
    if R_C_M <= 0 or U <= 0:
        return np.inf
    return tm_u + (U / R_C_M)

def compute_ttc(
    rhoT_p: Sequence[float],
    lambda_p: Sequence[float],
    eta: float,
    mu: float
) -> float:
    """
    Ttc = (sum_p rhoT_p * lambda_p / eta) / (mu - sum_p lambda_p)
    Conditions: mu - sum(lambda_p) > 0
    rhoT_p, lambda_p arrays must be same length P
    eta: average data size of computation tasks (bytes)
    mu: service rate of MEC server (bytes/sec of processing capacity)
    returns avg computation latency per byte (seconds/byte)
    """
    rhoT_p = np.asarray(rhoT_p, dtype=float)
    lambda_p = np.asarray(lambda_p, dtype=float)
    if eta <= 0:
        raise ValueError("eta must be > 0")
    denom = mu - np.sum(lambda_p)
    if denom <= 0:
        return np.inf
    numerator = np.sum(rhoT_p * lambda_p / eta)
    return numerator / denom

In [18]:
class LatencyModel:
    """
    LatencyModel computes per-tile latency and total latency for requests
    using the formulas from the paper (TD,U, TM,U, TC,U, Ttc and D matrix).

    P: number of DUs (p in paper)
    U: number of users
    R_M_D: total MEC->DU data rate (bytes/sec)
    R_C_M: total Cloud->MEC data rate (bytes/sec)
    mu: MEC service rate (bytes/sec) for transcoding
    eta: average computation task size (bytes)
    B_pu_matrix: shape (P, U) bandwidths from DU p to user u (bytes/sec). If None, set to 1e6
    gamma_pu_matrix: shape (P, U) SNRs. If None, set to 10 (10 linear)
    rhoT_p: length P list of proportion of transcoding tasks requested at CU
    lambda_p: length P list of arrival rates (requests/sec) from each DU
    """
    def __init__(
        self,
        P: int,
        U: int,
        R_M_D: float,
        R_C_M: float,
        mu: float,
        eta: float,
        B_pu_matrix: np.ndarray,
        gamma_pu_matrix: np.ndarray,
        rhoT_p: Sequence[float],
        lambda_p: Sequence[float],
        du_fixed_delay: float,
        mec_fixed_delay: float,
        cloud_fixed_delay: float
    ):
        self.P = P
        self.U = U
        self.R_M_D = float(R_M_D)
        self.R_C_M = float(R_C_M)
        self.mu = float(mu)
        self.eta = float(eta)

        self.du_fixed_delay = float(du_fixed_delay)
        self.mec_fixed_delay = float(mec_fixed_delay)
        self.cloud_fixed_delay = float(cloud_fixed_delay)

        self.B_pu = np.asarray(B_pu_matrix, dtype=float).reshape(P, U)
        self.gamma_pu = np.asarray(gamma_pu_matrix, dtype=float).reshape(P, U)
        self.rhoT_p = np.asarray(rhoT_p, dtype=float)
        self.lambda_p = np.asarray(lambda_p, dtype=float)

        # precompute TD,U
        self.TD_U = np.zeros((P, U), dtype=float)
        for p in range(P):
            for u in range(U):
                self.TD_U[p, u] = compute_td_u(
                    self.B_pu[p, u], 
                    self.gamma_pu[p, u]
                )

    def time_vector(self, p: int, u: int) -> Tuple[float, float, float, float, float]:
        td_u = self.TD_U[p, u]
        tm_u = compute_tm_u(td_u, self.R_M_D, self.U)
        tc_u = compute_tc_u(tm_u, self.R_C_M, self.U)
        ttc = compute_ttc(self.rhoT_p, self.lambda_p, self.eta, self.mu)
        
        return td_u, tm_u, tc_u, tm_u + ttc, tm_u + ttc

    def tile_latency(
        self, 
        p: int, 
        u: int, 
        tile_size_bytes: float, 
        events: Dict[str, int]
    ) -> float:
        """
        events dict must contain binary flags for:
         - 'alpha_p_u', 'alpha_M_u', 'alpha_C_u', 'beta_p_u', 'beta_M_u'
        Order corresponds to the 5 columns in the paper's D matrix.
        Returns latency in seconds for that tile (per tile_size_bytes).
        """
        T_vec = np.array(
            self.time_vector(p, u), 
            dtype=float
        )
        ES_vec = np.array([
            events.get("alpha_p_u", 0),
            events.get("alpha_M_u", 0),
            events.get("alpha_C_u", 0),
            events.get("beta_p_u", 0),
            events.get("beta_M_u", 0),
        ], dtype=float)
        
        per_byte_latency = float(np.dot(ES_vec, T_vec) * tile_size_bytes)

        # add fixed propagation delay(s) - only once per tile when DU or MEC are involved
        fixed = 0.0
        # if served from DU (column alpha_p_u) -> add DU fixed delay
        if events.get("alpha_p_u", 0) == 1 or events.get("beta_p_u", 0) == 1:
            fixed += self.du_fixed_delay
        # if served via MEC (alpha_M_u or beta_M_u) -> add MEC fixed delay
        if events.get("alpha_M_u", 0) == 1 or events.get("beta_M_u", 0) == 1:
            fixed += self.mec_fixed_delay
        # alpha_C_u (cloud) may also include additional fixed delays you can add as needed
        if events.get("alpha_C_u", 0) == 1:
            fixed += self.cloud_fixed_delay

        return per_byte_latency + fixed

    def total_request_latency(
        self, 
        p: int, 
        u: int, 
        tiles: Sequence[Dict[str, Any]]
    ) -> Tuple[float, Sequence[Tuple[str, float]]]:
      
        sorted_tiles = sorted(tiles, key=lambda t: t.get("layer", 0))
        
        total, tile_latencies = 0.0, []
        for tile in sorted_tiles:
            sz     = float(tile["size"])
            events = tile.get("events", {})
            tid    = tile.get("tile", None)
            layer  = tile.get("layer", 0)

            lat = self.tile_latency(p, u, sz, events)
            tile_latencies.append((tid, layer, lat))

            if total + lat >= 1.0:
                break
            
            total += lat
            
        return total, tile_latencies

    def compute_latency_for_request(self, request) -> Dict[str, Any]:
        p = request.get("du_id", 0)
        u = request.get("user_id", 0)
        tiles = request.get("tiles", [])

        total_latency, tile_latencies = self.total_request_latency(
            p, 
            u, 
            tiles
        )
        
        return {
            "total_latency": total_latency,
            "tile_latencies": tile_latencies
        }
    
    def user_link_throughput(
        self, 
        p: int, 
        u: int
    ) -> Dict[str, float]:
        """
        Returns per-user throughput (bytes/sec) for each stage:
        - 'du': wireless DU->user
        - 'mec': DU->user + MEC->DU (shared)
        - 'cloud': DU->user + MEC->DU + Cloud->MEC (shared)
        """
        td_u = self.TD_U[p, u]                         # sec/byte
        tm_u = compute_tm_u(td_u, self.R_M_D, self.U)  # sec/byte
        tc_u = compute_tc_u(tm_u, self.R_C_M, self.U)  # sec/byte

        def inv(x): return 0.0 if not np.isfinite(x) or x <= 0 else (1.0 / x)

        return {
            "du": inv(td_u),        # bytes/sec over DU->user link
            "mec": inv(tm_u),       # bytes/sec including MEC->DU
            "cloud": inv(tc_u)      # bytes/sec including Cloud->MEC
        }

    def request_effective_throughput(
        self, 
        p: int, 
        u: int, 
        tiles: Sequence[Dict[str, Any]]
    ) -> float:
        """
        Effective throughput for a request = total_bytes / total_time.
        Uses tile_latency() to accumulate time over all tiles.
        """
        total_bytes = 0.0
        total_time = 0.0
        for tile in tiles:
            sz = float(tile["size"])
            events = tile.get("events", {})
            total_bytes += sz
            total_time += self.tile_latency(p, u, sz, events)
        
        if total_time <= 0 or not np.isfinite(total_time):
            return 0.0
        
        return total_bytes / total_time

In [19]:
if __name__ == "__main__":
    # Create latency model (replace numbers with your real config)
    P = 1; U = 30
    lat_model = LatencyModel(
        P=P, 
        U=U,
        R_M_D=80e6,       # 640 Mbps -> 80e6 B/s 
        R_C_M=1.25e9,     # 10 Gbps -> 1.25e9 B/s
        mu=2e7, 
        eta=2e5,
        B_pu_matrix=np.full((P, U), 40e6, dtype=float),       # 320 Mbps -> 40e6 B/s
        gamma_pu_matrix=np.full((P, U), 5.0, dtype=float),    # SNRs
        rhoT_p=[0.2], 
        lambda_p=[0.05],
        du_fixed_delay=0.001,    # 1 ms
        mec_fixed_delay=0.005,   # 5 ms
        cloud_fixed_delay=0.05   # 50 ms
    )

    print(
        f"Experiment ===================\n"
        f"Total users: {U}\n"
        f"Latency Model Parameters:\n"
        f"  R_M_D = {lat_model.R_M_D} B/s\n"
        f"  R_C_M = {lat_model.R_C_M} B/s\n"
        f"  mu = {lat_model.mu} B/s\n"
        f"  eta = {lat_model.eta} B\n"
        f"  B_pu = {lat_model.B_pu[0,0]} B/s\n"
        f"  gamma_pu = {lat_model.gamma_pu[0,0]}\n"
        f"  rhoT_p = {lat_model.rhoT_p}\n"
        f"  lambda_p = {lat_model.lambda_p}\n"
        f"==============================\n"
    )

    request = {
        "seq": 0,
        "p": 0,
        "u": 0,
        "video": 0,
        "tiles": [
            {
                "tile": 1,
                "layer": 0,
                "size": 12000,
                "events": {
                    "alpha_p_u":1,
                    "alpha_M_u":0,
                    "alpha_C_u":0,
                    "beta_p_u":0,
                    "beta_M_u":0
                }
            },
            {
                "tile": 2, 
                "layer": 0,
                "size": 12000,
                "events": {
                    "alpha_p_u":0,
                    "alpha_M_u":1,
                    "alpha_C_u":0,
                    "beta_p_u":0,
                    "beta_M_u":0
                }
            },
            {
                "tile": 3, 
                "layer": 0,
                "size": 12000,
                "events": {
                    "alpha_p_u":0,
                    "alpha_M_u":0,
                    "alpha_C_u":1,
                    "beta_p_u":0,
                    "beta_M_u":0
                }
            },
        ]
    }

    latency_info = lat_model.compute_latency_for_request(request)

    print("User link throughput (bytes/sec):")
    for u in range(U):
        tp = lat_model.user_link_throughput(p=0, u=u)
        print(
            f" User {u}:"
            f" DU link: {tp['du']:.2f} B/s,"
            f" MEC link: {tp['mec']:.2f} B/s,"
            f" Cloud link: {tp['cloud']:.2f} B/s"
        )

Experiment ===================
Total users: 30
Latency Model Parameters:
  R_M_D = 80000000.0 B/s
  R_C_M = 1250000000.0 B/s
  mu = 20000000.0 B/s
  eta = 200000.0 B
  B_pu = 40000000.0 B/s
  gamma_pu = 5.0
  rhoT_p = [0.2]
  lambda_p = [0.05]

User link throughput (bytes/sec):
 User 0: DU link: 103398500.03 B/s, MEC link: 2599621.93 B/s, Cloud link: 2446954.19 B/s
 User 1: DU link: 103398500.03 B/s, MEC link: 2599621.93 B/s, Cloud link: 2446954.19 B/s
 User 2: DU link: 103398500.03 B/s, MEC link: 2599621.93 B/s, Cloud link: 2446954.19 B/s
 User 3: DU link: 103398500.03 B/s, MEC link: 2599621.93 B/s, Cloud link: 2446954.19 B/s
 User 4: DU link: 103398500.03 B/s, MEC link: 2599621.93 B/s, Cloud link: 2446954.19 B/s
 User 5: DU link: 103398500.03 B/s, MEC link: 2599621.93 B/s, Cloud link: 2446954.19 B/s
 User 6: DU link: 103398500.03 B/s, MEC link: 2599621.93 B/s, Cloud link: 2446954.19 B/s
 User 7: DU link: 103398500.03 B/s, MEC link: 2599621.93 B/s, Cloud link: 2446954.19 B/s
 User 8: 